In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error


# 결과 재현을 위한 난수 고정
rng = np.random.default_rng(42)


# 90일 분량의 시간대별 데이터 생성
n_hours = 24 * 90

time_index = pd.date_range(
    start="2026-01-01",
    periods=n_hours,
    freq="h"
)

df = pd.DataFrame(index=time_index)


# 시간 정보 생성
df["hour"] = df.index.hour
df["dayofweek"] = df.index.dayofweek
df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)


# 가상의 전력 사용량 생성
daily_pattern = 40 * np.sin(
    2 * np.pi * (df["hour"] - 7) / 24
)

weekday_effect = np.where(
    df["is_weekend"] == 0,
    20,
    -10
)

trend = np.linspace(0, 15, n_hours)
noise = rng.normal(0, 5, n_hours)

df["power"] = (
    200
    + daily_pattern
    + weekday_effect
    + trend
    + noise
)


# 과거 전력 사용량을 입력 변수로 사용
df["lag_1"] = df["power"].shift(1)
df["lag_24"] = df["power"].shift(24)
df["lag_168"] = df["power"].shift(168)

# 현재 시점 이전 24시간 평균
df["rolling_24_mean"] = (
    df["power"]
    .shift(1)
    .rolling(24)
    .mean()
)


# 시간을 원형 특성으로 변환
df["hour_sin"] = np.sin(
    2 * np.pi * df["hour"] / 24
)

df["hour_cos"] = np.cos(
    2 * np.pi * df["hour"] / 24
)


# 결측치가 있는 초기 행 제거
model_df = df.dropna().copy()


features = [
    "hour",
    "dayofweek",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "lag_1",
    "lag_24",
    "lag_168",
    "rolling_24_mean",
]

target = "power"


# 시간순 80% 학습, 20% 평가
split_index = int(len(model_df) * 0.8)

train_df = model_df.iloc[:split_index]
test_df = model_df.iloc[split_index:]

X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]


# LightGBM 모델 학습
model = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.03,
    num_leaves=31,
    random_state=42,
    verbosity=-1,
)

model.fit(X_train, y_train)


# 예측
predictions = model.predict(X_test)


# 평가
mae = mean_absolute_error(
    y_test,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        predictions
    )
)

print("학습 데이터:", X_train.shape)
print("평가 데이터:", X_test.shape)
print(f"MAE: {mae:.3f}")
print(f"RMSE: {rmse:.3f}")


# 마지막 7일 시각화
plot_hours = 24 * 7

plt.figure(figsize=(15, 5))

plt.plot(
    y_test.index[-plot_hours:],
    y_test.iloc[-plot_hours:],
    label="Actual",
)

plt.plot(
    y_test.index[-plot_hours:],
    predictions[-plot_hours:],
    label="Prediction",
    alpha=0.8,
)

plt.title("Hourly Power Forecast Practice")
plt.xlabel("Time")
plt.ylabel("Power")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import pandas as pd

data_dir = Path.home() / "projects" / "kamp-ai" / "data" / "raw" / "task5"

files = list(data_dir.rglob("*"))

for file in files:
    if file.is_file():
        print(file)

In [ ]:
from pathlib import Path
import pandas as pd

data_dir = (
    Path.home()
    / "projects"
    / "kamp-ai"
    / "data"
    / "raw"
    / "task5"
)

csv_path = next(
    data_dir.rglob("okm_augumented_2021.csv")
)

print("읽을 파일:", csv_path)


# 한글 CSV 인코딩을 순서대로 시도
encodings = [
    "utf-8-sig",
    "utf-8",
    "cp949",
    "euc-kr",
]

df = None

for encoding in encodings:
    try:
        df = pd.read_csv(
            csv_path,
            encoding=encoding
        )

        print("사용된 인코딩:", encoding)
        break

    except UnicodeDecodeError:
        print("인코딩 실패:", encoding)


if df is None:
    raise RuntimeError(
        "지원한 인코딩으로 CSV를 읽지 못했습니다."
    )


print("\n데이터 크기:", df.shape)

print("\n컬럼 목록:")
print(df.columns.tolist())

print("\n앞부분:")
display(df.head())

print("\n뒷부분:")
display(df.tail())

In [ ]:
print("=== 데이터 타입 ===")
df.info()

print("\n=== 컬럼별 결측치 ===")
display(
    df.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

print("\n전체 행 중복:", df.duplicated().sum())

print("\n=== 기초 통계 ===")
display(
    df.describe(
        include="all"
    ).T
)

In [ ]:
import numpy as np
import pandas as pd


audit_df = df.copy()


# 날짜 변환
audit_df["날짜_dt"] = pd.to_datetime(
    audit_df["날짜"].astype(str),
    format="%Y%m%d",
    errors="coerce",
)


print("=== 기간 ===")
print("시작:", audit_df["날짜_dt"].min())
print("종료:", audit_df["날짜_dt"].max())
print("날짜 개수:", audit_df["날짜_dt"].nunique())


print("\n=== 날짜 변환 실패 ===")
print(audit_df["날짜_dt"].isna().sum())


print("\n=== 하루당 행 개수 분포 ===")

daily_counts = (
    audit_df.groupby("날짜_dt")
    .size()
)

print(
    daily_counts
    .value_counts()
    .sort_index()
    .to_string()
)


print("\n24행이 아닌 날짜:")

not_24 = daily_counts[daily_counts != 24]

if len(not_24) == 0:
    print("없음")
else:
    print(not_24.to_string())


print("\n=== 비정상 시간 행 ===")

bad_hour_mask = ~audit_df["시간"].between(0, 23)

bad_hours = audit_df.loc[
    bad_hour_mask,
    ["날짜", "시간", "평균", "생산량"]
]

if len(bad_hours) == 0:
    print("없음")
else:
    print(bad_hours.to_string(index=False))


print("\n비정상 시간 행 개수:", bad_hour_mask.sum())


print("\n=== 0~23시 구성이 깨진 날짜 ===")

bad_days = []

for date, group in audit_df.groupby(
    "날짜_dt",
    sort=True
):
    hours = sorted(
        group["시간"].tolist()
    )

    if hours != list(range(24)):
        bad_days.append(
            {
                "날짜": date,
                "행 개수": len(group),
                "시간값": hours,
            }
        )

if len(bad_days) == 0:
    print("없음")
else:
    for item in bad_days:
        print(item)


print("\n=== 결측치가 있는 행 ===")

missing_rows = audit_df.loc[
    audit_df.isna().any(axis=1),
    [
        "날짜",
        "시간",
        "풍속",
        "강수량",
        "공장인원",
    ],
]

print(missing_rows.to_string(index=False))


print("\n=== 평균 계산 관계 ===")

quarter_columns = [
    "15분",
    "30분",
    "45분",
    "60분",
]

calculated_mean = (
    audit_df[quarter_columns]
    .mean(axis=1)
)

half_up_mean = np.floor(
    calculated_mean + 0.5
).astype(int)

numpy_round_mean = np.round(
    calculated_mean
).astype(int)

floor_mean = np.floor(
    calculated_mean
).astype(int)


print(
    "반올림한 평균과 일치율:",
    (audit_df["평균"] == half_up_mean).mean(),
)

print(
    "NumPy round와 일치율:",
    (audit_df["평균"] == numpy_round_mean).mean(),
)

print(
    "버림한 평균과 일치율:",
    (audit_df["평균"] == floor_mean).mean(),
)


print("\n=== 주요 컬럼 고유값 ===")

for column in [
    "day",
    "d",
    "m",
    "인건비",
    "전기요금(계절)",
]:
    values = sorted(
        audit_df[column]
        .dropna()
        .unique()
        .tolist()
    )

    print(f"{column}: {values[:30]}")

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


clean_df = df.copy()


# 날짜 변환
clean_df["날짜_dt"] = pd.to_datetime(
    clean_df["날짜"].astype(str),
    format="%Y%m%d",
)


# 원본 행 순서를 기준으로 각 날짜의 0~23시 생성
expected_hour = (
    clean_df.groupby(
        "날짜",
        sort=False
    )
    .cumcount()
)


# 정상 시간 행에서 원본 시간과 행 순서가 일치하는지 검사
valid_hour_mask = clean_df["시간"].between(0, 23)

hour_agreement = (
    clean_df.loc[
        valid_hour_mask,
        "시간"
    ].to_numpy()
    ==
    expected_hour.loc[
        valid_hour_mask
    ].to_numpy()
).mean()

print(
    "정상 행의 시간 순서 일치율:",
    hour_agreement
)

if hour_agreement != 1.0:
    raise ValueError(
        "정상 시간과 행 순서가 일치하지 않아 "
        "자동 복구를 중단합니다."
    )


# 원본 시간 보존
clean_df["시간_원본"] = clean_df["시간"]


# 모든 날짜의 시간을 행 순서 기준 0~23으로 복구
clean_df["시간"] = expected_hour.astype(int)


# 완전한 시각 생성
clean_df["timestamp"] = (
    clean_df["날짜_dt"]
    + pd.to_timedelta(
        clean_df["시간"],
        unit="h"
    )
)


# 시간순 정렬
clean_df = (
    clean_df
    .sort_values("timestamp")
    .reset_index(drop=True)
)


# 1시간 간격 연속성 검사
time_diff = (
    clean_df["timestamp"]
    .diff()
    .dropna()
)

print(
    "모든 행이 1시간 간격:",
    time_diff.eq(
        pd.Timedelta(hours=1)
    ).all()
)


# 결측치 여부를 별도 변수로 보존
missing_columns = [
    "풍속",
    "강수량",
    "공장인원",
]

for column in missing_columns:
    clean_df[
        f"{column}_결측"
    ] = (
        clean_df[column]
        .isna()
        .astype(int)
    )


# 시계열 순서를 이용한 선형 보간
clean_df[missing_columns] = (
    clean_df[missing_columns]
    .interpolate(
        method="linear",
        limit_direction="both"
    )
)


print(
    "보간 후 전체 결측치:",
    clean_df.isna().sum().sum()
)


# 날짜 파생 컬럼 검증
day_match = (
    clean_df["day"]
    ==
    clean_df["날짜_dt"].dt.dayofweek + 1
).mean()

date_match = (
    clean_df["d"]
    ==
    clean_df["날짜_dt"].dt.day
).mean()

month_match = (
    clean_df["m"]
    ==
    clean_df["날짜_dt"].dt.month
).mean()

print("day와 요일 일치율:", day_match)
print("d와 일자 일치율:", date_match)
print("m과 월 일치율:", month_match)


# 평균값 관계를 기록하되 모델 입력에는 사용하지 않음
quarter_columns = [
    "15분",
    "30분",
    "45분",
    "60분",
]

clean_df["평균_재계산"] = np.floor(
    clean_df[quarter_columns]
    .mean(axis=1)
    + 0.5
).astype(int)

print(
    "평균 재계산 일치율:",
    (
        clean_df["평균"]
        ==
        clean_df["평균_재계산"]
    ).mean()
)


print("\n복구된 문제 날짜:")

print(
    clean_df.loc[
        clean_df["날짜"].isin(
            [20210713, 20210715]
        ),
        [
            "날짜",
            "시간_원본",
            "시간",
            "timestamp",
            "평균",
        ],
    ].to_string(index=False)
)

In [ ]:
processed_dir = (
    Path.home()
    / "projects"
    / "kamp-ai"
    / "data"
    / "processed"
    / "task5"
)

processed_dir.mkdir(
    parents=True,
    exist_ok=True
)

processed_path = (
    processed_dir
    / "task5_clean.csv"
)

clean_df.to_csv(
    processed_path,
    index=False,
    encoding="utf-8-sig",
)

print("저장 완료:")
print(processed_path)